# Notebook 02: Image Preprocessing and Dual Data Augmentation

**Objective:** Build, test, and visually inspect the preprocessing and augmentation pipeline for industrial defect inspection.

### Core Concepts Explored:
1. **BGR vs RGB Color Spaces:** Why OpenCV's default BGR scrambles convolutional feature detectors trained on RGB.
2. **Standardization & Invertibility:** Why $\mu = [0.485, 0.456, 0.406], \sigma = [0.229, 0.224, 0.225]$ are required for transfer learning, and how to invert them for visual reporting.
3. **Discrete Mask Preservation:** Nearest-neighbor interpolation vs bilinear blurring.
4. **Synchronized Dual Augmentation:** Applying identical geometric transformations to image and segmentation mask simultaneously.

In [ ]:
import sys
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch

# Ensure project root is in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import MVTEC_BOTTLE_DIR
from src.data.preprocessing import (
    load_image,
    to_tensor,
    normalize_tensor,
    denormalize_tensor,
    tensor_to_numpy_image,
    load_and_preprocess_mask,
    IMAGENET_MEAN,
    IMAGENET_STD
)
from src.data.augmentation import IndustrialDualAugmentation

print("Preprocessing modules loaded successfully.")

## 1. Color Space Inspection: BGR vs RGB

OpenCV reads images in **BGR** order by default. Matplotlib and deep learning backbones expect **RGB** order.
Let's see what happens if BGR is passed directly to an RGB viewer.

In [ ]:
sample_path = next(MVTEC_BOTTLE_DIR.glob("test/broken_large/*.png"))

img_bgr = load_image(sample_path, target_size=(300, 300), color_space="BGR")
img_rgb = load_image(sample_path, target_size=(300, 300), color_space="RGB")

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_bgr)
axes[0].set_title("Raw OpenCV BGR (Colors Inverted)")
axes[0].axis("off")

axes[1].imshow(img_rgb)
axes[1].set_title("Converted to RGB (Correct)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 2. Standardization & Invertibility (Denormalization)

Standardization calculates $z = \frac{x - \mu}{\sigma}$.
To display model predictions or overlay heatmaps on user images, we must reconstruct the original image via:
$$x = z \cdot \sigma + \mu$$
Let's verify that the round-trip reconstruction error is zero.

In [ ]:
tensor_raw = to_tensor(img_rgb)
tensor_norm = normalize_tensor(tensor_raw, IMAGENET_MEAN, IMAGENET_STD)
tensor_reconstructed = denormalize_tensor(tensor_norm, IMAGENET_MEAN, IMAGENET_STD)
img_reconstructed = tensor_to_numpy_image(tensor_reconstructed)

max_error = np.abs(img_rgb.astype(float) - img_reconstructed.astype(float)).max()
print(f"Maximum pixel reconstruction error: {max_error:.6f}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(img_rgb)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(tensor_norm[0].numpy(), cmap="coolwarm")
axes[1].set_title("Normalized Tensor (Red Channel)")
axes[1].axis("off")

axes[2].imshow(img_reconstructed)
axes[2].set_title("Inverted / Reconstructed")
axes[2].axis("off")
plt.tight_layout()
plt.show()

## 3. Discrete Mask Nearest-Neighbor Resizing

Verify that nearest-neighbor interpolation preserves discrete binary values $\{0.0, 1.0\}$ without fractional blurring.

In [ ]:
mask_path = MVTEC_BOTTLE_DIR / "ground_truth" / "broken_large" / f"{sample_path.stem}_mask.png"
mask_tensor = load_and_preprocess_mask(mask_path, target_size=(256, 256))

unique_values = torch.unique(mask_tensor).tolist()
print(f"Unique values in discrete mask tensor: {unique_values}")
assert unique_values == [0.0, 1.0], "Mask contains non-binary interpolated values!"

plt.figure(figsize=(4, 4))
plt.imshow(mask_tensor[0].numpy(), cmap="gray")
plt.title("Nearest-Neighbor Binarized Mask (256x256)")
plt.axis("off")
plt.show()

## 4. Dual Augmentation Pipeline (Image & Mask Synchronized)

We test `IndustrialDualAugmentation` to demonstrate that:
1. Rotation and horizontal flips are locked synchronously between image and mask.
2. Photometric jitter (brightness/contrast) changes image pixels while keeping mask values binary.

In [ ]:
aug_pipe = IndustrialDualAugmentation(
    target_size=(384, 384),
    mode="train",
    hflip_prob=0.5,
    max_rotation_degrees=12.0,
    brightness_jitter=0.2,
    contrast_jitter=0.2,
    normalize=False
)

fig, axes = plt.subplots(3, 3, figsize=(11, 10))

for row in range(3):
    a_img_t, a_mask_t = aug_pipe(tensor_raw.clone(), mask_tensor.clone())
    a_img = tensor_to_numpy_image(a_img_t)
    a_mask = (a_mask_t[0].numpy() > 0).astype(np.uint8)
    
    # Color overlay in translucent red
    overlay = a_img.copy()
    overlay[a_mask > 0] = [255, 30, 30]
    blended = cv2.addWeighted(a_img, 0.65, overlay, 0.35, 0)
    
    axes[row, 0].imshow(a_img)
    axes[row, 0].set_title(f"Augmented Image #{row+1}")
    axes[row, 0].axis("off")
    
    axes[row, 1].imshow(a_mask, cmap="gray")
    axes[row, 1].set_title(f"Synchronized Mask #{row+1}")
    axes[row, 1].axis("off")
    
    axes[row, 2].imshow(blended)
    axes[row, 2].set_title(f"Synchronized Overlay #{row+1}")
    axes[row, 2].axis("off")

plt.suptitle("Dual Augmentation: Verification of Spatial Synchronization", fontsize=13)
plt.tight_layout()
plt.show()